<a href="https://colab.research.google.com/github/San-Maansson/Machine_Learning/blob/main/neural_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Neural Network
"""

!pip install tensorflow
!pip install panel functions
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

from panel_functions import split_panel

In [ ]:
#============================================================
#STEP 1: Training and testing data
#============================================================
merged = pd.read_pickle('merged_data.pkl')

#Explain excess returns with factors
y = merged['ER']
X = merged.drop(columns=['ER'])

#In and out-of-sample: train-test split
#Relative size of out-of-sample period: e.g., 0.1 is 10% of data
y_train, y_test = split_panel(y)
X_train, X_test = split_panel(X)


In [ ]:
#============================================================
#STEP 2: Model evaluation
#============================================================
def evaluate_model(name, model, X_train, X_test, y_train, y_test, is_nn=False):
    if is_nn:
        # Scale features for neural nets
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
        y_pred = model.predict(X_test).flatten()
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{name}: R2 = {r2:.3f}, RMSE = {rmse:.3f}")

In [ ]:
#============================================================
#STEP 3: OLS Regression, Random Forests, and Neural Networks
#============================================================
#OLS Regression
ols = LinearRegression()
evaluate_model("OLS Regression", ols, X_train, X_test, y_train, y_test)

#Random Forests
rf = RandomForestRegressor(n_estimators=200, random_state=42)
evaluate_model("Random Forest", rf, X_train, X_test, y_train, y_test)

#Neural Networks
nn = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)
])
nn.compile(optimizer=Adam(learning_rate=0.01), loss='mse')

evaluate_model("Neural Net", nn, X_train, X_test, y_train, y_test, is_nn=True)